# Plot Light Curves vs MJD — Multiple Flux Types Comparison

For each stable star in `data_MergeVisits_02_out/per_star/`, this notebook
produces a **GridSpec 6×2 figure** per star showing all available flux types
overlaid on the same subplots:

- **Left column (6 rows)** — all flux types vs `expMidptMJD` per band (u g r i z y),
  colour-coded by flux type.  `psfFlux` is the **reference** (drawn in red, always
  on top).  Aperture fluxes (`ap09`, `ap12`, `ap17`, `ap25`, `ap35`) are drawn with
  a perceptually-uniform colour palette.  `calibFlux` is shown in dark grey.
  The y-axis is clipped to ±N_SIGMA_YLIM × σ_IQR(psfFlux) so all flux types are
  directly comparable on the same scale.

- **Right column (6 rows)** — overlaid histograms of all flux types with individual
  Gaussian fits.  `psfFlux` is plotted at the foreground with a solid black fit
  curve; other flux types are plotted translucently behind it.

**Section 8** — Stacked normalised residuals `(f − median) / median` for *each*
flux type in a 2×3 panel grid (one panel per band), allowing a direct comparison
of the photometric precision of each measurement method.

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Université Paris-Saclay
- **Created:** 2026-06-29
- **Reference notebook:** `03_PlotLCwithMJD.ipynb`


## 1. Imports

In [ ]:
import gc
import logging
import os
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MultipleLocator, AutoMinorLocator
from matplotlib.ticker import MaxNLocator

from astropy.time import Time
from scipy.optimize import curve_fit

In [ ]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → %matplotlib inline")

## 2. Logging

In [ ]:
log = logging.getLogger()
log.setLevel(logging.INFO)
if not log.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)
    log.addHandler(handler)
log.info("Logging initialised.")

## 3. Configuration

In [ ]:
# ── Notebook tag ──────────────────────────────────────────────────────
NB_TAG = "PlotLC_04"

# ── Input: merged LC files from notebook 02 ───────────────────────────
DIR_DATA_IN = "./data_MergeVisits_02_out"
DIR_PER_STAR_IN = os.path.join(DIR_DATA_IN, "per_star")

# ── Output figures ────────────────────────────────────────────────────
DIR_FIGS = f"./figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
log.info("Figure directory: %s", DIR_FIGS)

# ── Common columns ────────────────────────────────────────────────────
MJD_COL = "expMidptMJD"
BAND_COL = "band"

# ── Flux types to display: (column_name, error_column, label, color) ──
# psfFlux is the REFERENCE: drawn first (foreground), red colour.
# Aperture radii in pixels: ap09=0.9", ap12=1.2", ap17=1.7", ap25=2.5", ap35=3.5"
# calibFlux is the calibrated flux (photometric calibration applied at catalog level).
FLUX_TYPES = [
    # (flux_col,       err_col,          label,            color,       zorder, lw)
    ("psfFlux", "psfFluxErr", "psfFlux (ref)", "#D62728", 5, 1.2),  # red — reference
    ("calibFlux", "calibFluxErr", "calibFlux", "#2C2C2C", 4, 0.9),  # dark grey
    ("ap09Flux", "ap09FluxErr", "ap09Flux", "#1F77B4", 3, 0.8),  # blue
    ("ap12Flux", "ap12FluxErr", "ap12Flux", "#FF7F0E", 3, 0.8),  # orange
    ("ap17Flux", "ap17FluxErr", "ap17Flux", "#2CA02C", 3, 0.8),  # green
    ("ap25Flux", "ap25FluxErr", "ap25Flux", "#9467BD", 3, 0.8),  # purple
    ("ap35Flux", "ap35FluxErr", "ap35Flux", "#8C564B", 3, 0.8),  # brown
]

# Reference flux used to set the y-axis range (σ_IQR scaling)
REF_FLUX_COL = "psfFlux"
REF_FLUX_ERR_COL = "psfFluxErr"

# ── Band ordering and colours (LSST ugrizy) ───────────────────────────
BAND_ORDER = ["u", "g", "r", "i", "z", "y"]
BAND_COLORS = {
    "u": "#6C2DC7",
    "g": "#1CB94A",
    "r": "#E8262A",
    "i": "#E87E26",
    "z": "#C95C93",
    "y": "#994D00",
}

# ── Y-axis clipping (based on psfFlux σ_IQR) ─────────────────────────
N_SIGMA_YLIM = 5.0

# Minimum number of valid psfFlux points to attempt any plotting
MIN_POINTS = 5

# ── Figure parameters ─────────────────────────────────────────────────
FIG_WIDTH = 18
FIG_HEIGHT = 16

## 4. Helper functions

In [ ]:
def sigma_iqr(values):
    """Robust σ estimate via the inter-quartile range: σ_IQR = IQR / 1.3490."""
    q75, q25 = np.nanpercentile(values, [75, 25])
    return (q75 - q25) / 1.3489795


def gaussian(x, mu, sigma, amplitude):
    """1-D Gaussian for curve_fit."""
    return amplitude * np.exp(-0.5 * ((x - mu) / sigma) ** 2)


def fit_gaussian(values, n_bins=30, flux_range=None):
    """Fit a Gaussian to a histogram of *values*.

    Returns
    -------
    (mu, sigma, amplitude, bin_centers, hist_counts)  or
    (None, None, None, bin_centers, hist_counts)  on failure.
    """
    kw = {"bins": n_bins}
    if flux_range is not None:
        kw["range"] = flux_range
    counts, edges = np.histogram(values, **kw)
    centers = 0.5 * (edges[:-1] + edges[1:])

    mu0 = np.median(values)
    sig0 = sigma_iqr(values)
    amp0 = counts.max()
    if sig0 <= 0 or amp0 <= 0:
        return None, None, None, centers, counts

    try:
        popt, _ = curve_fit(gaussian, centers, counts, p0=[mu0, sig0, amp0], maxfev=5000)
        mu_f, sig_f, amp_f = popt
        if sig_f <= 0 or abs(mu_f - mu0) > 5 * sig0:
            return None, None, None, centers, counts
        return abs(mu_f), abs(sig_f), abs(amp_f), centers, counts
    except RuntimeError:
        return None, None, None, centers, counts


def savefig(fig, name, dpi=150):
    """Save *fig* as both PDF and PNG under DIR_FIGS."""
    base = os.path.join(DIR_FIGS, name)
    fig.savefig(base + ".pdf", dpi=dpi, bbox_inches="tight")
    fig.savefig(base + ".png", dpi=dpi, bbox_inches="tight")
    log.info("Saved: %s (.pdf/.png)", base)


def sigma_nJy_to_mmag(sigma_nJy, median_nJy):
    """Convert flux scatter σ (nJy) to mmag using the linearised formula."""
    if median_nJy <= 0 or sigma_nJy < 0:
        return np.nan
    return 1000.0 * (2.5 / np.log(10)) * (sigma_nJy / abs(median_nJy))


def available_flux_types(df):
    """Return the subset of FLUX_TYPES whose flux column exists in *df*."""
    return [(fc, ec, lbl, col, zo, lw) for fc, ec, lbl, col, zo, lw in FLUX_TYPES if fc in df.columns]

## 5. Core plotting function: one star → one 6×2 GridSpec figure

In [ ]:
def plot_star_multiflux(df_star: pd.DataFrame, simbad_id: str, file_stem: str) -> None:
    """Build and save the 6×2 multi-flux GridSpec figure for one stable star.

    Left panels : all available flux types vs MJD, colour-coded per flux type.
                  psfFlux is the reference (red, foreground).
    Right panels: overlaid histograms + Gaussian fits for each flux type.
                  psfFlux histogram is drawn in the foreground.

    The y-axis range of every light-curve panel is set from psfFlux only
    (median ± N_SIGMA_YLIM × σ_IQR) so that all flux types are on the same scale.

    Parameters
    ----------
    df_star   : merged LC DataFrame for this star.
    simbad_id : human-readable identifier shown in the figure title.
    file_stem : output filename stem (no extension, no directory).
    """
    # Drop rows with NaN in the reference flux or MJD
    df = df_star.dropna(subset=[MJD_COL, REF_FLUX_COL]).copy()

    # Mean (ra, dec)
    ra = df["ra"].mean() if "ra" in df.columns else float("nan")
    dec = df["dec"].mean() if "dec" in df.columns else float("nan")

    # Which flux types are present in this dataframe?
    avail_flux = available_flux_types(df)

    # ── Global MJD range (shared x-axis) ──────────────────────────────
    valid_mjd_mask = df[MJD_COL].notna()
    if REF_FLUX_ERR_COL in df.columns:
        valid_mjd_mask &= df[REF_FLUX_ERR_COL].notna() & (df[REF_FLUX_ERR_COL] > 0)
    valid_mjd_all = df.loc[valid_mjd_mask, MJD_COL]

    if len(valid_mjd_all) > 0:
        global_mjd_min = valid_mjd_all.min()
        global_mjd_max = valid_mjd_all.max()
    else:
        global_mjd_min, global_mjd_max = 0.0, 1.0

    mjd_pad = max((global_mjd_max - global_mjd_min) * 0.005, 0.5)
    global_mjd_min -= mjd_pad
    global_mjd_max += mjd_pad

    mjd_range = global_mjd_max - global_mjd_min
    n_ticks = min(5, max(2, int(mjd_range // 30) + 2))
    global_tick_mjd = np.linspace(global_mjd_min, global_mjd_max, n_ticks)
    global_tick_dates = Time(global_tick_mjd, format="mjd").to_value("iso", subfmt="date")

    # ── Build flux-type legend handles (for figure-level legend) ───────
    flux_legend_handles = []
    flux_legend_labels = []

    # ── Figure & GridSpec ──────────────────────────────────────────────
    fig = plt.figure(figsize=(FIG_WIDTH, FIG_HEIGHT))
    flux_labels_str = ", ".join(lbl for _, _, lbl, _, _, _ in avail_flux)
    fig.suptitle(
        f"{simbad_id}  |  (ra,dec)=({ra:.3f},{dec:.3f})\n"
        f"Multi-flux light curves per band  |  y-axis: psfFlux median ± {N_SIGMA_YLIM:.0f}×σ_IQR\n"
        f"Flux types shown: {flux_labels_str}",
        fontsize=12,
        fontweight="bold",
        y=0.95,
    )

    gs = gridspec.GridSpec(
        6,
        2,
        figure=fig,
        width_ratios=[2, 1],
        hspace=0.60,
        wspace=0.28,
    )

    for row_idx, band in enumerate(BAND_ORDER):
        band_color = BAND_COLORS.get(band, "steelblue")

        ax_lc = fig.add_subplot(gs[row_idx, 0])
        ax_hist = fig.add_subplot(gs[row_idx, 1])

        # ── Select band rows, require valid reference flux + error ─────
        mask_ref = (df[BAND_COL] == band) & df[MJD_COL].notna() & df[REF_FLUX_COL].notna()
        if REF_FLUX_ERR_COL in df.columns:
            mask_ref &= df[REF_FLUX_ERR_COL].notna() & (df[REF_FLUX_ERR_COL] > 0)

        grp_ref = df[mask_ref].sort_values(MJD_COL)
        n_pts = len(grp_ref)

        # ── Empty band ─────────────────────────────────────────────────
        if n_pts < MIN_POINTS:
            for ax in (ax_lc, ax_hist):
                ax.text(
                    0.5,
                    0.5,
                    f"{band}  —  {n_pts} point(s)",
                    ha="center",
                    va="center",
                    transform=ax.transAxes,
                    color="grey",
                    fontsize=10,
                )
                ax.set_xticks([])
                ax.set_yticks([])
            ax_lc.set_xlim(global_mjd_min, global_mjd_max)
            continue

        # Reference flux statistics (used for y-axis scaling)
        ref_flux = grp_ref[REF_FLUX_COL].values
        med_ref = np.nanmedian(ref_flux)
        sig_ref = sigma_iqr(ref_flux)
        if sig_ref == 0:
            sig_ref = np.nanstd(ref_flux) or 1.0

        ylim_lo = med_ref - N_SIGMA_YLIM * sig_ref
        ylim_hi = med_ref + N_SIGMA_YLIM * sig_ref
        if ylim_hi - ylim_lo < 1e-10 * abs(med_ref) + 1.0:
            ylim_lo = med_ref - 1.0
            ylim_hi = med_ref + 1.0

        mjd_ref = grp_ref[MJD_COL].values

        # ── LEFT panel — multi-flux light curves ──────────────────────
        # Draw non-reference flux types first (background)
        for fc, ec, lbl, col, zo, lw in avail_flux:
            if fc == REF_FLUX_COL:
                continue  # reference drawn last (foreground)
            flux_vals = grp_ref[fc].values if fc in grp_ref.columns else None
            if flux_vals is None or np.all(np.isnan(flux_vals)):
                continue
            err_vals = grp_ref[ec].values if (ec in grp_ref.columns) else np.zeros_like(flux_vals)
            ax_lc.errorbar(
                mjd_ref,
                flux_vals,
                yerr=err_vals,
                fmt="o",
                ms=2.5,
                color=col,
                ecolor=col,
                alpha=0.55,
                elinewidth=0.6,
                capsize=1.5,
                lw=lw,
                zorder=zo,
                label=lbl,
            )

        # Reference (psfFlux) — foreground
        ref_err = (
            grp_ref[REF_FLUX_ERR_COL].values
            if REF_FLUX_ERR_COL in grp_ref.columns
            else np.zeros_like(ref_flux)
        )
        ax_lc.errorbar(
            mjd_ref,
            ref_flux,
            yerr=ref_err,
            fmt="o",
            ms=4.0,
            color="#D62728",
            ecolor="#D62728",
            alpha=0.85,
            elinewidth=0.9,
            capsize=2.0,
            lw=1.2,
            zorder=5,
            label="psfFlux (ref)",
        )

        # Median reference line + σ_IQR shaded band
        ax_lc.axhline(med_ref, color="#D62728", lw=1.2, ls="--", alpha=0.8, zorder=4)
        ax_lc.axhspan(med_ref - sig_ref, med_ref + sig_ref, color="#D62728", alpha=0.07, zorder=1)

        ax_lc.set_ylim(ylim_lo, ylim_hi)
        ax_lc.set_xlim(global_mjd_min, global_mjd_max)

        rel_scatter = 100.0 * sig_ref / abs(med_ref) if abs(med_ref) > 0 else np.nan
        ax_lc.text(
            0.02,
            0.96,
            f"psfFlux: σ_IQR = {sig_ref:.1f} nJy  ({rel_scatter:.2f}%)",
            transform=ax_lc.transAxes,
            fontsize=8,
            va="top",
            ha="left",
            color="#D62728",
        )

        ax_lc.set_ylabel(f"{band}  flux [nJy]", fontsize=9, color=band_color)
        ax_lc.tick_params(axis="y", labelsize=8)
        ax_lc.tick_params(axis="x", labelsize=7)
        ax_lc.yaxis.set_minor_locator(AutoMinorLocator())
        ax_lc.tick_params(which="minor", length=2)

        # Secondary top x-axis (calendar date)
        ax_top = ax_lc.twiny()
        ax_top.set_xlim(global_mjd_min, global_mjd_max)
        ax_top.set_xticks(global_tick_mjd)
        ax_top.set_xticklabels(global_tick_dates, fontsize=7, rotation=20, ha="left")
        ax_top.tick_params(length=3)

        if row_idx == len(BAND_ORDER) - 1:
            ax_lc.set_xlabel("expMidptMJD", fontsize=9)

        ax_lc.legend(loc="upper right", fontsize=7, framealpha=0.6, ncol=2, handlelength=1.5)
        ax_lc.grid(True, lw=0.4, alpha=0.4)

        # ── RIGHT panel — overlaid histograms ─────────────────────────
        ax_hist.set_title(f"{band} band", fontsize=9, color=band_color)

        # Background flux types (not reference)
        for fc, ec, lbl, col, zo, lw in avail_flux:
            if fc == REF_FLUX_COL:
                continue
            fvals = grp_ref[fc].values if fc in grp_ref.columns else None
            if fvals is None or np.all(np.isnan(fvals)):
                continue
            fvals = fvals[np.isfinite(fvals)]
            if len(fvals) < 3:
                continue
            mu_f, sig_f, amp_f, bc, hc = fit_gaussian(fvals, flux_range=(ylim_lo, ylim_hi))
            ax_hist.bar(
                bc,
                hc,
                width=(bc[1] - bc[0]) * 0.9,
                color=col,
                alpha=0.35,
                edgecolor="none",
                label=lbl,
                zorder=zo - 2,
            )
            if mu_f is not None:
                xf = np.linspace(bc[0], bc[-1], 300)
                ax_hist.plot(xf, gaussian(xf, mu_f, sig_f, amp_f), color=col, lw=1.2, ls="--", zorder=zo)

        # Reference histogram (psfFlux) — foreground
        ref_clean = ref_flux[np.isfinite(ref_flux)]
        mu_psf, sig_psf, amp_psf, bc_psf, hc_psf = fit_gaussian(ref_clean, flux_range=(ylim_lo, ylim_hi))
        ax_hist.bar(
            bc_psf,
            hc_psf,
            width=(bc_psf[1] - bc_psf[0]) * 0.9,
            color="#D62728",
            alpha=0.60,
            edgecolor="none",
            label="psfFlux (ref)",
            zorder=4,
        )
        if mu_psf is not None:
            xf = np.linspace(bc_psf[0], bc_psf[-1], 300)
            ax_hist.plot(
                xf, gaussian(xf, mu_psf, sig_psf, amp_psf), color="k", lw=1.8, zorder=5, label="psf Gauss"
            )
            ax_hist.axvline(mu_psf, color="k", lw=1.2, ls="--")
            ax_hist.axvline(mu_psf - sig_psf, color="grey", lw=0.9, ls=":")
            ax_hist.axvline(mu_psf + sig_psf, color="grey", lw=0.9, ls=":")

            rel_psf = 100.0 * sig_psf / mu_psf if mu_psf > 0 else np.nan
            mmag_psf = 1000.0 * (2.5 / np.log(10)) * sig_psf / mu_psf if mu_psf > 0 else np.nan
            ax_hist.text(
                0.97,
                0.96,
                f"psf: μ={mu_psf:.0f}\n" f"  σ={sig_psf:.1f} nJy\n" f"  {rel_psf:.2f}%  {mmag_psf:.0f} mmag",
                transform=ax_hist.transAxes,
                fontsize=7,
                va="top",
                ha="right",
                color="k",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.75),
            )

        ax_hist.set_xlim(ylim_lo, ylim_hi)
        ax_hist.set_xlabel("flux [nJy]", fontsize=8)
        ax_hist.set_ylabel("counts", fontsize=8)
        ax_hist.tick_params(labelsize=8)
        ax_hist.legend(fontsize=6, loc="upper left", framealpha=0.6, ncol=1, handlelength=1.2)
        ax_hist.grid(True, lw=0.3, alpha=0.35)

    savefig(fig, file_stem)
    gc.collect()

## 6. Discover per-star files and run the plotting loop

In [ ]:
lc_files = sorted(f for f in os.listdir(DIR_PER_STAR_IN) if f.endswith("_lc_mjd.csv"))
log.info("Found %d per-star LC files.", len(lc_files))
lc_files

In [ ]:
n_ok = 0
n_err = 0

for fname in lc_files:
    src_path = os.path.join(DIR_PER_STAR_IN, fname)

    try:
        df_star = pd.read_csv(src_path)
    except Exception as exc:
        log.error("  ERROR reading %s: %s", fname, exc)
        n_err += 1
        continue

    simbad_id = (
        df_star["simbad_id"].iloc[0]
        if "simbad_id" in df_star.columns and len(df_star) > 0
        else fname.replace("_lc_mjd.csv", "")
    )
    file_stem = fname.replace("_lc_mjd.csv", "") + "_LC_MJD_multiflux"

    log.info("Plotting multiflux: %s  (%d rows)", simbad_id, len(df_star))

    try:
        plot_star_multiflux(df_star, simbad_id, file_stem)
        n_ok += 1
    except Exception as exc:
        log.error("  ERROR plotting %s: %s", simbad_id, exc)
        plt.close("all")
        n_err += 1

log.info("Done — %d figures saved, %d errors.", n_ok, n_err)

## 7. Quick inline preview

In [ ]:
from IPython.display import Image, display

saved_pngs = sorted(
    os.path.join(DIR_FIGS, f) for f in os.listdir(DIR_FIGS) if f.endswith(".png") and "multiflux" in f
)

if saved_pngs:
    log.info("Previewing: %s", saved_pngs[0])
    display(Image(filename=saved_pngs[0], width=950))
else:
    log.warning("No multiflux PNG found in %s", DIR_FIGS)

## 8. Normalised flux residuals per flux type — 2×3 panel comparison

For each LSST band (u g r i z y), one 2×3 panel figure is produced showing
the stacked normalised residuals

$$\delta = \frac{f - \tilde{f}}{\tilde{f}}$$

for **each flux type** overlaid as semi-transparent histograms with
individual Gaussian fits.  This gives a direct, per-band comparison of the
photometric precision of `psfFlux`, `calibFlux`, `ap09Flux`, …, `ap35Flux`.

> **psfFlux** is drawn in red with a black Gaussian curve; other flux types
> follow the colour scheme defined in `FLUX_TYPES`.

In [ ]:
# ── Load global merged CSV ────────────────────────────────────────────
global_csv = os.path.join(DIR_DATA_IN, "all_stars_lightcurves_mjd.csv")
df_all = pd.read_csv(global_csv)
df_all = df_all.dropna(subset=[MJD_COL, REF_FLUX_COL])
if REF_FLUX_ERR_COL in df_all.columns:
    df_all = df_all[df_all[REF_FLUX_ERR_COL] > 0]

all_stars = sorted(df_all["simbad_id"].unique())
avail_flux_global = available_flux_types(df_all)

N_BINS_RESID = 40
RESID_XLIM = 0.20  # ±20%

log.info(
    "Global CSV loaded: %d rows, %d stars, flux types: %s",
    len(df_all),
    len(all_stars),
    [lbl for _, _, lbl, _, _, _ in avail_flux_global],
)

In [ ]:
# ── One 2×3 figure: residuals for ALL flux types per band ─────────────
#
# Layout:  2 rows of 3 band subplots  +  1 narrow legend row

fig_res = plt.figure(figsize=(16, 10))
fig_res.suptitle(
    r"Normalised flux residuals  $\delta = (f - \tilde{f}) / \tilde{f}$  per band"
    " — all flux types overlaid\n"
    "Stacked over all stable stars  |  colour per flux type  |  Gaussian fits",
    fontsize=13,
    fontweight="bold",
    y=0.98,
)

gs_res = gridspec.GridSpec(
    3,
    3,
    figure=fig_res,
    hspace=0.52,
    wspace=0.35,
    height_ratios=[1, 1, 0.22],
)

band_pos = {"u": (0, 0), "g": (0, 1), "r": (0, 2), "i": (1, 0), "z": (1, 1), "y": (1, 2)}

# ── Summary table data ────────────────────────────────────────────────
summary_rows = []

for band in BAND_ORDER:
    row, col = band_pos[band]
    band_color = BAND_COLORS.get(band, "steelblue")
    ax = fig_res.add_subplot(gs_res[row, col])

    # Draw each flux type
    for fc, ec, lbl, col_flux, zo, lw in avail_flux_global:
        is_ref = fc == REF_FLUX_COL

        all_resid = []
        for sid in all_stars:
            mask = (df_all["simbad_id"] == sid) & (df_all[BAND_COL] == band) & df_all[fc].notna()
            sub = df_all.loc[mask, fc].values
            if len(sub) < MIN_POINTS:
                continue
            med_s = np.nanmedian(sub)
            if abs(med_s) < 1e-12:
                continue
            all_resid.append((sub - med_s) / abs(med_s))

        if not all_resid:
            continue

        all_r = np.concatenate(all_resid)

        # Histogram
        counts_h, edges_h = np.histogram(all_r, bins=N_BINS_RESID, range=(-RESID_XLIM, RESID_XLIM))
        centers_h = 0.5 * (edges_h[:-1] + edges_h[1:])

        ax.bar(
            centers_h,
            counts_h,
            width=(centers_h[1] - centers_h[0]) * 0.85,
            color=col_flux,
            alpha=0.65 if is_ref else 0.35,
            edgecolor="none",
            zorder=zo if is_ref else zo - 2,
            label=lbl,
        )

        # Gaussian fit
        sig0 = sigma_iqr(all_r)
        sig0 = sig0 if sig0 > 0 else 0.01
        try:
            popt, _ = curve_fit(
                gaussian,
                centers_h,
                counts_h,
                p0=[np.nanmedian(all_r), sig0, counts_h.max()],
                maxfev=5000,
            )
            mu_g, sig_g, amp_g = popt
            sig_g = abs(sig_g)
            if sig_g > 0 and abs(mu_g) < 5 * sig0:
                xg = np.linspace(-RESID_XLIM, RESID_XLIM, 400)
                # Reference: thick black curve; others: coloured dashed
                ax.plot(
                    xg,
                    gaussian(xg, mu_g, sig_g, amp_g),
                    color="k" if is_ref else col_flux,
                    lw=2.0 if is_ref else 1.2,
                    ls="-" if is_ref else "--",
                    zorder=zo + 1,
                )
                if is_ref:
                    ax.axvline(mu_g, color="k", lw=1.2, ls="--", zorder=zo + 1)

                summary_rows.append(
                    {
                        "band": band,
                        "flux_type": lbl,
                        "N_total": len(all_r),
                        "sigma_fit_%": round(sig_g * 100.0, 3),
                        "sigma_fit_mmag": round(1000.0 * (2.5 / np.log(10)) * sig_g, 1),
                        "mu_fit_%": round(mu_g * 100.0, 3),
                    }
                )
        except (RuntimeError, ValueError):
            pass

    # ── Reference lines ───────────────────────────────────────────────
    for ref_val, ls_, alpha_ in [(0.01, ":", 0.5), (0.05, "--", 0.4), (0.10, "-.", 0.3)]:
        for sign in (-1, 1):
            ax.axvline(sign * ref_val, color="grey", lw=0.8, ls=ls_, alpha=alpha_)

    ax.set_xlim(-RESID_XLIM, RESID_XLIM)
    ax.set_xlabel(r"$(f - \tilde{f}) / \tilde{f}$", fontsize=9)
    ax.set_ylabel("counts", fontsize=9)
    ax.tick_params(labelsize=8)
    ax.grid(True, lw=0.3, alpha=0.35)
    ax.set_title(f"{band} band", fontsize=11, color=band_color, fontweight="bold")
    ax.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(xmax=1.0, decimals=0))
    ax.legend(fontsize=7, loc="upper left", framealpha=0.6, ncol=1, handlelength=1.2)

# ── Bottom legend row (flux-type colour key) ──────────────────────────
ax_leg = fig_res.add_subplot(gs_res[2, :])
ax_leg.axis("off")
from matplotlib.patches import Patch

leg_handles = [
    Patch(facecolor=col_flux, alpha=0.7, label=lbl) for _, _, lbl, col_flux, _, _ in avail_flux_global
]
ax_leg.legend(
    leg_handles,
    [lbl for _, _, lbl, _, _, _ in avail_flux_global],
    loc="upper center",
    ncol=len(avail_flux_global),
    fontsize=9,
    frameon=True,
    framealpha=0.85,
    title="Flux types",
    title_fontsize=9,
    handlelength=1.5,
    handleheight=0.9,
    borderpad=0.6,
    columnspacing=1.0,
)

savefig(fig_res, "residuals_multiflux_all_bands_2x3")
plt.show()
log.info("Multi-flux residuals figure saved.")

## 9. Precision summary table: σ_fit per (band × flux type)

In [ ]:
# ── Build the precision table from summary_rows ───────────────────────
if summary_rows:
    df_prec = pd.DataFrame(summary_rows)

    # Sort by band order and flux type order defined in FLUX_TYPES
    band_ord_map = {b: i for i, b in enumerate(BAND_ORDER)}
    flux_ord_map = {lbl: i for i, (_, _, lbl, _, _, _) in enumerate(FLUX_TYPES)}
    df_prec["_b"] = df_prec["band"].map(band_ord_map)
    df_prec["_f"] = df_prec["flux_type"].map(flux_ord_map).fillna(99)
    df_prec = df_prec.sort_values(["_b", "_f"]).drop(columns=["_b", "_f"]).reset_index(drop=True)

    print(f"Precision summary — {len(df_prec)} (band × flux_type) entries")
    display(df_prec)
else:
    print("No precision data accumulated (all fits failed).")
    df_prec = pd.DataFrame()

In [ ]:
# ── Pivot table: sigma_fit_% indexed by flux_type (rows) × band (cols) ──
if not df_prec.empty:
    pivot = df_prec.pivot_table(
        index="flux_type",
        columns="band",
        values="sigma_fit_%",
        aggfunc="first",
    )
    # Reorder columns and rows
    pivot = pivot.reindex(columns=[b for b in BAND_ORDER if b in pivot.columns])
    flux_order_list = [lbl for _, _, lbl, _, _, _ in FLUX_TYPES if lbl in pivot.index]
    pivot = pivot.reindex(index=flux_order_list)

    print("\nσ_fit (%) per flux type × band:")
    display(pivot.style.background_gradient(cmap="RdYlGn_r", axis=None).format("{:.2f}", na_rep="--"))

    # Save
    prec_out = os.path.join(DIR_FIGS, "precision_multiflux_summary.csv")
    df_prec.to_csv(prec_out, index=False)
    log.info("Precision table saved: %s", prec_out)

In [ ]:
# ── Bar-chart comparison of sigma_fit_% across flux types for each band ──
if not df_prec.empty:
    bands_present = [b for b in BAND_ORDER if b in df_prec["band"].values]
    n_bands = len(bands_present)
    n_cols = 3
    n_rows = int(np.ceil(n_bands / n_cols))

    fig_bar, axes_bar = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows), squeeze=False)
    fig_bar.suptitle(
        r"$\sigma_{\rm fit}$ (%) per flux type and band" " — photometric precision comparison",
        fontsize=13,
        fontweight="bold",
        y=1.001,
    )

    for idx, band in enumerate(bands_present):
        ax_b = axes_bar[idx // n_cols][idx % n_cols]
        sub = df_prec[df_prec["band"] == band].copy()

        colors_bar = [
            next((col_flux for _, _, lbl, col_flux, _, _ in avail_flux_global if lbl == ft), "steelblue")
            for ft in sub["flux_type"]
        ]

        bars = ax_b.bar(
            sub["flux_type"],
            sub["sigma_fit_%"],
            color=colors_bar,
            alpha=0.80,
            edgecolor="k",
            linewidth=0.5,
        )
        # Annotate bars with mmag value
        for bar_rect, (_, row_data) in zip(bars, sub.iterrows()):
            ax_b.text(
                bar_rect.get_x() + bar_rect.get_width() / 2,
                bar_rect.get_height() + 0.01 * bar_rect.get_height() + 0.005,
                f"{row_data['sigma_fit_mmag']:.0f} mmag",
                ha="center",
                va="bottom",
                fontsize=7,
                rotation=0,
            )

        ax_b.set_title(f"{band} band", color=BAND_COLORS.get(band, "k"), fontsize=11, fontweight="bold")
        ax_b.set_ylabel(r"$\sigma_{\rm fit}$ (%)", fontsize=9)
        ax_b.tick_params(axis="x", labelsize=7, rotation=25)
        ax_b.tick_params(axis="y", labelsize=8)
        ax_b.grid(axis="y", lw=0.4, alpha=0.4)
        ax_b.set_ylim(bottom=0)

    # Hide unused subplots
    for idx in range(len(bands_present), n_rows * n_cols):
        axes_bar[idx // n_cols][idx % n_cols].set_visible(False)

    fig_bar.tight_layout()
    savefig(fig_bar, "precision_bar_chart_multiflux")
    plt.show()
    log.info("Precision bar-chart saved.")